# Inference and context-aware gene sets tutorial

This tutorial outlines how to run inference using a trained Multicellular Transformer model on the AnnData object.
We show how to set up a datamodule, investigate the output and extract context-aware gene sets.

Before starting, make sure that you have the repo installed.

#### Import the dependencies

In [1]:
import torch
import scanpy as sc
import json
import numpy as np

from lightning import seed_everything
from typing import Dict

from cell_cell_interactions.modeling.model import MulticellularTransformerPLModel
from cell_cell_interactions.pp.adata_rnaseq_datamodule import AdataRnaseqDataModule
from cell_cell_interactions.tl.grn import GeneEmbedding
from cell_cell_interactions.pl.grn import plot_gene_set


seed_everything(32)

[rank: 0] Seed set to 32


32

For this dataset you will also need a GPU, as running Multicellular Transformer without it takes a very long time

In [2]:
# assert GPU is available
assert torch.cuda.is_available()

## Read and preprocess lung cancer atlas

For inference, we will use the lung cancer atlas [High-resolution single-cell atlas reveals diversity and plasticity of tumor-associated neutrophils in non-small cell lung cancer](https://www.sciencedirect.com/science/article/pii/S1535610822004998?via%3Dihub) which can be downloaded from CELL x GENE [here](https://datasets.cellxgene.cziscience.com/c366b76c-b5cf-499b-ad99-84bd57f71f0f.h5ad).

We then preprocess the dataset removing cells and genes with very little counts, followed by standard normalization.

#### If you are a Genentech employee, you can skip this step and read in already preprocessed data

#### Reading and preprocessing the dataset

In [ ]:
adata = sc.read_h5ad("<input-dir>/c366b76c-b5cf-499b-ad99-84bd57f71f0f.h5ad")

In [ ]:
# filter lowly expressed genes and cells
sc.pp.filter_genes(adata, min_counts=5)
sc.pp.filter_cells(adata, min_counts=100)
# normalize counts in a cell to sum to 10000
sc.pp.normalize_total(adata, total_sum=10000)
# perform log1p normalization
sc.pp.log1p(adata)

# print adata details
adata

#### Write the dataset, as the DataModule requires a path to AnnData

In [ ]:
adata.write_h5ad("<your-desired-location>")

## Load the data into a datamodule

Use the datamodule for Adata object to read the object and preprocess it for data loading.

In [5]:
# replace with your own path
adata_file_path = "/data/bucket/wiatrakm/cell-cell-interactions/input-data/lung-cancer-atlas/lung_cancer_atlas_processed_all_genes_log1pnorm.h5ad"
# the global gene2idx from pretraining
gene2idx_file_path = "data/ensemble2idx.json"

with open(gene2idx_file_path) as f:
    gene2idx = json.load(f)

# get the data
datamodule = AdataRnaseqDataModule(
    adata_file_path=adata_file_path,  # filepath to anndata object with preprocess lung cancer atlas
    gene2idx=gene2idx,  # global gene2idx mapping
    val_set_proportion=0.2,
    sample_size=512,  # nr of cells 
    sample_column="sample",  # the column name of the sample 
    label_column="cell_type_major",  # the cell type column
    tissue_column="tissue",  # the tissue column
    min_sample_size=512,  # min nr of cells per sample for the sample to be considered
    min_cell_type_size=0.002,   # min relative proportion of the cell type out of all cells to be considers, otherwise we filter it out
    sample_sampling="weighted",  # whether to sample samples considering the nr of cells in a sample or not ("random")
    cell_sampling="stratified",  # whether to use stratify cell type sampling, by default "stratified"
    batch_size=1,  # batch size
    num_workers=4,  # nr of workers for the dataloader
    agg_cell_type_gex=False,  # whether to use pseudobulked expression aggregated by cell type
    mask_prob=0.,  # not do any masking
    perc_min_expressed_cells_per_gene=0.01,  # filter out genes which are expressed in less then 1% of cells in a sample
    max_seq_len=12000,   # max nr of genes in a sample (after filtering out the lowly expressed genes)
    random_seed=32,  # random state
    backed=True,  # whether to read AnnData object in backed mode, recommended to use it
)

Training data size: (1026803, 53)
Validation data size: (231608, 53)


#### Fetch a dataloader and investigate its outputs

In [6]:
val_dataloader = datamodule.val_dataloader()
batch = next(iter(val_dataloader))

/homefs/home/wiatrakm/miniforge3/envs/ccc/lib/python3.10/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [7]:
# investigate the batch
batch

BagOfCellsBatch(expression_matrix=tensor([[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.4599],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]]]), gene_ids=tensor([[ 5162, 48836, 13953,  ..., 10701, 10380,  1275]]), labels=tensor([[ 5162, 48836, 13953,  ..., 10701, 10380,  1275]]), is_masked_gene_id=tensor([[0, 0, 0,  ..., 0, 0, 0]]), is_masked_gene_expression=tensor([[0, 0, 0,  ..., 0, 0, 0]]), tissue_labels=tensor([-100]), padding_mask=tensor([[False, False, False,  ..., False, False, False]]), tissue_idx=tensor([2]), cell_type_indices=None, study_label=None, sample_label=['Madissoon_Meyer_2020_HCATisStab7587202'], disease=['normal'], n_expressed_genes=tensor([11502.]), cell_indices=None, cell_i

## Load the model

Load the model, here we use a model finetuned on the lung cancer data.

In [8]:
ckpt_path = "/data/bucket/wiatrakm/cell-cell-interactions/output-data/runs-lung-cancer-finetuning/lung_cancer_no_finetune/checkpoints/epoch=0-step=20000-val_total_loss=4.98.ckpt"
model = MulticellularTransformerPLModel.load_from_checkpoint(ckpt_path, strict=True).cuda()
model.eval()

MulticellularTransformerPLModel(
  (model): MulticellularTransformerModule(
    (encoder): MulticellularTransformerEncoder(
      (gene_id_emb): Embedding(58056, 512, padding_idx=0)
      (tissue_emb): Embedding(7, 512, padding_idx=0)
      (gene_expression_encoder): Sequential(
        (0): Linear(in_features=512, out_features=512, bias=True)
        (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (2): ReLU()
        (3): Dropout(p=0.2, inplace=False)
        (4): Linear(in_features=512, out_features=512, bias=True)
      )
      (gene_expression_mask_emb): Embedding(1, 512)
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
      (encoder): TransformerEncoder(
        (layers): ModuleList(
          (0-11): 12 x TransformerEncoderLayer(
            (self_attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
            )

## Perform inference

Using the datamodule and trained model, perform inference

In [10]:
with torch.no_grad():
    output = model.model(
        expression_matrix=batch.expression_matrix.to(model.device),
        gene_ids=batch.gene_ids.to(model.device),
        is_masked_gene_expression=batch.is_masked_gene_expression.to(model.device),
        padding_mask=batch.padding_mask.to(model.device),
        tissue=batch.tissue_idx.to(model.device),
    )

# investigate last_hidden_state shape
print("Shape of last hidden state:", output.last_hidden_state.shape)

Shape of last hidden state: torch.Size([1, 11502, 512])


## Extract context-aware gene sets

Using the contextual gene embeddings extracted from the output of the last layer of transformer, perform Leiden clustering to identify gene sets.
These gene sets can be then used to analyse the pathways present in the sample.

In [11]:
# get sample details
print(f'Analysing sample with disease: {batch.disease[0]}, with {batch.gene_ids.shape[-1]} genes. Sample ID: {batch.sample_label[0]}')

Analysing sample with disease: normal, with 11502 genes. Sample ID: Madissoon_Meyer_2020_HCATisStab7587202


#### Fetch gene embeddings

In [12]:
def get_gene_embeddings(
    gene_ids: torch.Tensor,
    last_hidden_state: torch.Tensor,
    idx2gene_symbol: Dict[int, str], 
) -> Dict[str, np.ndarray]:
    """Process the gene ids and last hidden state to a dictionary used for gene sets extraction."""
    gene_embeddings = {}
    for idx in range(gene_ids.shape[-1]):
        gene_id = batch.gene_ids[:, idx].item()
        if gene_id not in idx2gene_symbol:  # account for PAD, SEP, CLS token
            continue
        gene_symbol = idx2gene_symbol[gene_id]
        gene_embeddings[gene_symbol] = last_hidden_state[:, idx, :].squeeze(0).numpy()

    return gene_embeddings


with open("data/gene_symbol2idx.json", "r") as f:
    gene_symbol2idx = json.load(f)
idx2gene_symbol = {idx: gene_symbol for gene_symbol, idx in gene_symbol2idx.items()}

gene_embeddings = get_gene_embeddings(
    gene_ids=batch.gene_ids,
    last_hidden_state=output.last_hidden_state.cpu(),
    idx2gene_symbol=idx2gene_symbol,
)

## Perform Leiden clustering

In [ ]:
# get GeneEmbedding object
embed = GeneEmbedding(gene_embeddings)

# Perform Leiden clustering with desired resolution
# lower resolutions are gonna give larger gene sets
# higher resolutions are gonna give smaller, more specific gene sets
gdata = embed.get_adata(resolution=40)
# Retrieve the gene clusters
gene_sets = embed.get_metagenes(gdata)
# filter gene sets with nr of genes lower than 4
gene_sets = {gid: gs for gid, gs in gene_sets.items() if len(gs) >= 4}
gene_set_sizes = [len(gs) for gs in gene_sets.values()]
print(f"Nr of gene sets: {len(gene_sets)}, average gene set size: {np.mean(gene_set_sizes)}, " 
      f"std gene set size: {np.std(gene_set_sizes)}")

100%|█████████████████████████████████████████████████████████████████████████████████████| 11501/11501 [00:00<00:00, 2054283.72it/s]


#### Plot a gene set of interest

In [45]:
# select a gene set with EGFR gene
egfr_gene_set = [gs for gs_id, gs in gene_sets.items() if 'EGFR' in gs][0]
plot_gene_set(egfr_gene_set, embed, thresh=0.55, seed=2)

['ACSS3',
 'CACNA1D',
 'COL8A1',
 'COPZ2',
 'DCBLD2',
 'EGFR',
 'EPHB3',
 'ERI2',
 'GLIS2',
 'GLIS3',
 'GPC1',
 'HDAC11',
 'KANK2',
 'LAMB2',
 'LAPTM4B',
 'LHFPL6',
 'LRIG3',
 'MTARC2',
 'NDNF',
 'NT5DC2',
 'NXN',
 'OSMR',
 'PHYHD1',
 'PRKD1',
 'PTPRU',
 'PYCR1',
 'RBMS3',
 'SGCE',
 'SMARCA1',
 'SNHG18',
 'TMEM30B',
 'TNKS1BP1',
 'YAP1']